# 01 — Sensor evaluation and target-variable selection

This notebook audits the original greenhouse observations and creates the validated sensor dataset used by notebooks 02–10. It examines the sensor-to-column mapping, temporal coverage, physical ranges, agreement between the SHT31 and BME280 sensors, and BME280 relative-humidity saturation.

**Inputs**

- `data/raw/greenhouse_sensor_data_raw.csv`

**Main outputs**

- `data/processed/greenhouse_sensor_data_validated.csv`
- Audit tables in `results/sensor_evaluation/`
- Diagnostic figures in PNG and PDF format in `figures/sensor_evaluation/`

All paths are relative to the repository root. Run the notebook with the environment defined in `requirements.txt`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

BME280_RH_SATURATION_THRESHOLD = 99.5

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from the repository root "
        "or from its notebooks directory."
    )

PROJECT_ROOT = find_project_root()
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "greenhouse_sensor_data_raw.csv"
PROCESSED_FILE = PROJECT_ROOT / "data" / "processed" / "greenhouse_sensor_data_validated.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "sensor_evaluation"
FIGURES_DIR = PROJECT_ROOT / "figures" / "sensor_evaluation"

PROCESSED_FILE.parent.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {RAW_FILE.relative_to(PROJECT_ROOT)}")


## 1. Load the observations and identify the sensor mapping

Column names are standardized only in memory. `Temp 1`, `Hum 1`, and `Presion` correspond to the BME280; `Temp 2` and `Hum 2` correspond to the SHT31.


In [ ]:
if not RAW_FILE.exists():
    raise FileNotFoundError(f"Required input file not found: {RAW_FILE}")

raw = pd.read_csv(RAW_FILE, encoding="utf-8-sig")
raw.columns = raw.columns.str.strip()

column_map = {
    "Fecha": "timestamp",
    "Temp 1": "temp_bme280",
    "Hum 1": "rh_bme280",
    "Presion": "pressure_bme280",
    "Temp 2": "temp_sht31",
    "Hum 2": "rh_sht31",
    "UV mW/cm²": "uv_mw_cm2",
    "UV mW/cmÂ²": "uv_mw_cm2",
}
required_raw_columns = {"Fecha", "Temp 1", "Hum 1", "Presion", "Temp 2", "Hum 2"}
missing_columns = sorted(required_raw_columns.difference(raw.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = raw.rename(columns={key: value for key, value in column_map.items() if key in raw.columns}).copy()
if "uv_mw_cm2" not in df.columns:
    df["uv_mw_cm2"] = np.nan

df["timestamp"] = pd.to_datetime(df["timestamp"], dayfirst=True, errors="coerce")
numeric_columns = ["temp_bme280", "rh_bme280", "pressure_bme280","temp_sht31", "rh_sht31", "uv_mw_cm2",]

df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)

sensor_mapping = pd.DataFrame(
    [
        ("Temp 1", "BME280", "temp_bme280"),
        ("Hum 1", "BME280", "rh_bme280"),
        ("Presion", "BME280", "pressure_bme280"),
        ("Temp 2", "SHT31", "temp_sht31"),
        ("Hum 2", "SHT31", "rh_sht31"),
    ],
    columns=["original_column", "sensor", "processed_column"],
)
sensor_mapping.to_csv(RESULTS_DIR / "01_sensor_column_mapping.csv", index=False)

print(f"Records loaded: {len(df):,}")
display(sensor_mapping)

## 2. Dataset audit

This section summarizes temporal coverage, sampling intervals, variable availability, and physically implausible observations. It does not impute, aggregate, or remove records.


In [ ]:
general_audit = pd.DataFrame(
    {
        "indicator": [
            "Number of records", "Initial timestamp", "Final timestamp",
            "Invalid timestamps", "Duplicate timestamps",
        ],
        "value": [
            len(df), df["timestamp"].min(), df["timestamp"].max(),
            int(df["timestamp"].isna().sum()),
            int(df["timestamp"].duplicated().sum()),
        ],
    }
)
general_audit.to_csv(RESULTS_DIR / "02_general_audit.csv", index=False)

unique_timestamps = df["timestamp"].dropna().drop_duplicates().sort_values()
interval_seconds = unique_timestamps.diff().dt.total_seconds().dropna()
interval_summary = pd.DataFrame(
    {
        "indicator": [
            "Median [s]", "Mean [s]", "Standard deviation [s]", "Minimum [s]",
            "5th percentile [s]", "25th percentile [s]", "75th percentile [s]",
            "95th percentile [s]", "Maximum [s]", "Percentage between 210 and 270 s",
        ],
        "value": [
            interval_seconds.median(), interval_seconds.mean(), interval_seconds.std(),
            interval_seconds.min(), interval_seconds.quantile(0.05), interval_seconds.quantile(0.25),
            interval_seconds.quantile(0.75), interval_seconds.quantile(0.95), interval_seconds.max(),
            interval_seconds.between(210, 270).mean() * 100,
        ],
    }
)
interval_summary.to_csv(RESULTS_DIR / "03_sampling_interval_summary.csv", index=False)


def summarize_variable(series):
    valid = series.dropna()
    return {
        "n_valid": int(valid.size),
        "n_missing": int(series.isna().sum()),
        "coverage_pct": float(series.notna().mean() * 100),
        "minimum": float(valid.min()) if not valid.empty else np.nan,
        "maximum": float(valid.max()) if not valid.empty else np.nan,
        "mean": float(valid.mean()) if not valid.empty else np.nan,
        "median": float(valid.median()) if not valid.empty else np.nan,
        "standard_deviation": float(valid.std()) if not valid.empty else np.nan,
        "n_unique": int(valid.nunique()),
        "zero_pct": float(valid.eq(0).mean() * 100) if not valid.empty else np.nan,
    }


coverage = pd.DataFrame(
    {column: summarize_variable(df[column]) for column in numeric_columns}
).T.reset_index(names="variable")
coverage.to_csv(RESULTS_DIR / "04_variable_coverage.csv", index=False)

physical_ranges = {
    "temp_sht31": (-40, 125),
    "rh_sht31": (0, 100),
    "temp_bme280": (-40, 85),
    "rh_bme280": (0, 100),
    "pressure_bme280": (300, 1100),
}
range_rows = []
for variable, (lower, upper) in physical_ranges.items():
    outside = df[variable].notna() & ~df[variable].between(lower, upper)
    range_rows.append(
        {
            "variable": variable,
            "lower_limit": lower,
            "upper_limit": upper,
            "n_outside_range": int(outside.sum()),
            "outside_range_pct": float(outside.mean() * 100),
        }
    )
range_audit = pd.DataFrame(range_rows)
range_audit.to_csv(RESULTS_DIR / "05_physical_range_audit.csv", index=False)

display(general_audit)
display(coverage[["variable", "n_valid", "n_missing", "coverage_pct", "minimum", "maximum"]])
display(range_audit)


## 3. Sensor agreement and BME280 relative-humidity saturation

Differences are defined as SHT31 minus BME280. BME280 relative-humidity observations at or above 99.5% RH are flagged as saturated. SHT31 temperature and relative humidity remain the forecasting targets; the quality-controlled BME280 measurements are retained only as candidate auxiliary predictors.


In [ ]:
def lins_ccc(x, y):
    pair = pd.DataFrame({"x": x, "y": y}).dropna()
    x_values = pair["x"].to_numpy(dtype=float)
    y_values = pair["y"].to_numpy(dtype=float)
    covariance = np.cov(x_values, y_values, ddof=1)[0, 1]
    denominator = (
        np.var(x_values, ddof=1)
        + np.var(y_values, ddof=1)
        + (x_values.mean() - y_values.mean()) ** 2
    )
    return float(2 * covariance / denominator) if denominator else np.nan


def agreement_metrics(reference, comparison, variable):
    pair = pd.DataFrame({"reference": reference, "comparison": comparison}).dropna()
    difference = pair["reference"] - pair["comparison"]
    bias = difference.mean()
    difference_sd = difference.std(ddof=1)
    pearson_r, pearson_p = stats.pearsonr(pair["reference"], pair["comparison"])
    spearman_rho, spearman_p = stats.spearmanr(pair["reference"], pair["comparison"])
    return {
        "variable": variable,
        "n_pairs": len(pair),
        "bias_sht31_minus_bme280": bias,
        "median_difference": difference.median(),
        "mae_between_sensors": difference.abs().mean(),
        "rmse_between_sensors": np.sqrt(np.mean(np.square(difference))),
        "difference_standard_deviation": difference_sd,
        "lower_95_agreement_limit": bias - 1.96 * difference_sd,
        "upper_95_agreement_limit": bias + 1.96 * difference_sd,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        "lins_ccc": lins_ccc(pair["reference"], pair["comparison"]),
    }


df["delta_temp"] = df["temp_sht31"] - df["temp_bme280"]
df["delta_rh"] = df["rh_sht31"] - df["rh_bme280"]
df["rh_bme280_raw"] = df["rh_bme280"]
df["rh_bme280_saturated"] = df["rh_bme280_raw"] >= BME280_RH_SATURATION_THRESHOLD
df["rh_bme280_clean"] = df["rh_bme280_raw"].mask(df["rh_bme280_saturated"])
df["rh_bme280_valid_for_model"] = df["rh_bme280_clean"].notna()

agreement = pd.DataFrame(
    [
        agreement_metrics(df["temp_sht31"], df["temp_bme280"], "air_temperature_degC"),
        agreement_metrics(df["rh_sht31"], df["rh_bme280"], "relative_humidity_pct"),
    ]
)
agreement.to_csv(RESULTS_DIR / "06_sensor_agreement.csv", index=False)

rh_quality = pd.DataFrame(
    {
        "indicator": [
            "Total records", "Records valid for modeling", "Saturated records",
            "Records invalid for modeling", "Global saturation [%]",
        ],
        "value": [
            len(df),
            int(df["rh_bme280_valid_for_model"].sum()),
            int(df["rh_bme280_saturated"].sum()),
            int((~df["rh_bme280_valid_for_model"]).sum()),
            float(df["rh_bme280_saturated"].mean() * 100),
        ],
    }
)
rh_quality.to_csv(RESULTS_DIR / "07_bme280_rh_quality.csv", index=False)

display(agreement)
display(rh_quality)


## 4. Validated dataset and diagnostic figures

The validated dataset preserves every historical record and adds explicit BME280 quality flags. Figures are exported in both PNG and PDF format for reuse in reports and publications.


In [ ]:
validated_columns = [
    "timestamp", "temp_sht31", "rh_sht31", "temp_bme280", "rh_bme280",
    "pressure_bme280", "rh_bme280_raw", "rh_bme280_clean",
    "rh_bme280_saturated", "rh_bme280_valid_for_model", "delta_temp", "delta_rh",
]
validated = df[validated_columns].copy()
validated["rh_bme280_saturated"] = validated["rh_bme280_saturated"].astype(int)
validated.to_csv(PROCESSED_FILE, index=False, encoding="utf-8")

sensor_selection = pd.DataFrame(
    [
        {
            "variable": "Air temperature",
            "target_sensor": "SHT31",
            "auxiliary_sensor": "BME280",
            "decision": "SHT31 selected as forecasting target",
            "rationale": "Better nominal specifications and high thermal agreement with the BME280.",
        },
        {
            "variable": "Relative humidity",
            "target_sensor": "SHT31",
            "auxiliary_sensor": "Quality-controlled BME280",
            "decision": "SHT31 selected as forecasting target",
            "rationale": "BME280 humidity shows time-dependent discrepancy and saturation periods.",
        },
        {
            "variable": "Atmospheric pressure",
            "target_sensor": "Not applicable",
            "auxiliary_sensor": "BME280",
            "decision": "Retained only as a candidate exogenous predictor",
            "rationale": "Its contribution is assessed through ablation and chronological validation.",
        },
    ]
)
sensor_selection.to_csv(RESULTS_DIR / "08_target_variable_selection.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
comparisons = [
    ("temp_bme280", "temp_sht31", "Air temperature (°C)"),
    ("rh_bme280", "rh_sht31", "Relative humidity (%RH)"),
]
for axis, (x_column, y_column, label) in zip(axes, comparisons):
    axis.scatter(df[x_column], df[y_column], s=6, alpha=0.20, rasterized=True)
    low = np.nanmin([df[x_column].min(), df[y_column].min()])
    high = np.nanmax([df[x_column].max(), df[y_column].max()])
    axis.plot([low, high], [low, high], color="black", linestyle="--", linewidth=1)
    axis.set_xlabel(f"BME280 {label}")
    axis.set_ylabel(f"SHT31 {label}")
    axis.set_title(f"Sensor agreement: {label}")
fig.tight_layout()
for extension in ("png", "pdf"):
    fig.savefig(FIGURES_DIR / f"01_sensor_agreement.{extension}", dpi=300, bbox_inches="tight")
plt.show()

fig, axis = plt.subplots(figsize=(12, 5))
axis.plot(df["timestamp"], df["rh_sht31"], linewidth=0.8, label="SHT31")
axis.plot(df["timestamp"], df["rh_bme280"], linewidth=0.7, alpha=0.65, label="BME280")
saturated = df["rh_bme280_saturated"]
axis.scatter(
    df.loc[saturated, "timestamp"],
    df.loc[saturated, "rh_bme280"],
    s=7,
    color="crimson",
    label="BME280 saturation flag",
    rasterized=True,
)
axis.set_xlabel("Date")
axis.set_ylabel("Relative humidity (%RH)")
axis.set_title("Relative-humidity observations and BME280 saturation")
axis.legend(loc="best")
fig.tight_layout()
for extension in ("png", "pdf"):
    fig.savefig(FIGURES_DIR / f"02_bme280_rh_saturation.{extension}", dpi=300, bbox_inches="tight")
plt.show()

display(sensor_selection)
print(f"Validated dataset: {PROCESSED_FILE.relative_to(PROJECT_ROOT)}")
print(f"Result tables: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Figures: {FIGURES_DIR.relative_to(PROJECT_ROOT)}")
